# Training Pipeline

Complete pipeline for:
1. Preprocessing PDB files into PyTorch graph objects
2. Creating train/val/test datasets with cross-validation splits
3. Setting up DataLoaders for training

## 1. Environment Setup

In [ ]:
# Uncomment for Colab
# !pip uninstall -y torch torchvision torchaudio -q
# !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128 -q
# !pip install fair-esm biopython torch_geometric -q
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.8.0+cu128.html -q
# !sudo apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq dssp
# !sudo ln -s /usr/bin/mkdssp /usr/bin/dssp

In [1]:
import sys
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from torch_geometric.loader import DataLoader

# Ensure src is importable
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

: 

## 2. Configuration

In [ ]:
# Paths
CSV_PATH = "data/pdb_splits.csv"
OUTPUT_DIR = "preprocessed"
RAW_PDB_DIR = "data/raw_pdbs"
ESM_CACHE_DIR = ".esm_cache"

# Processing settings
BATCH_SIZE = 10          # Proteins per batch for ESM (lower if OOM)
MAX_WORKERS = 8          # Parallel download workers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training settings
CV_FOLD = 1              # Validation fold (1-5)
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 32

print(f"Device: {DEVICE}")

## 3. Prepare CSV (Format Conversion)

Convert binding residues from `B_E103` (chain_AA+resseq) to `E103` format.

In [ ]:
def convert_binding_format(binding_str: str) -> str:
    """Convert 'B_E103 B_R45' to 'E103 R45' format."""
    if pd.isna(binding_str) or not binding_str.strip():
        return ""
    
    tokens = binding_str.strip().split()
    converted = []
    
    for token in tokens:
        if "_" in token:
            parts = token.split("_", 1)
            if len(parts) == 2:
                converted.append(parts[1])
        else:
            converted.append(token)
    
    return " ".join(converted)


# Load and convert CSV
df_raw = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_raw)} rows from {CSV_PATH}")

print(f"\nBefore: {df_raw['Binding_Residues'].iloc[0][:50]}...")
df_raw["Binding_Residues"] = df_raw["Binding_Residues"].apply(convert_binding_format)
print(f"After:  {df_raw['Binding_Residues'].iloc[0][:50]}...")

# Save converted CSV
CONVERTED_CSV = "data/pdb_splits_converted.csv"
df_raw.to_csv(CONVERTED_CSV, index=False)
print(f"\nSaved to {CONVERTED_CSV}")

In [ ]:
# Data summary
print("Data Summary")
print("=" * 40)
print(f"Total proteins: {len(df_raw)}")
print(f"\nSplit distribution:")
print(df_raw["Split"].value_counts())
print(f"\nCV Batch distribution (training):")
print(df_raw[df_raw["Split"] == "training"]["CV_Batch"].value_counts().sort_index())

## 4. Run Preprocessing Pipeline

Downloads PDBs, computes ESM2 embeddings, builds graphs, saves `.pt` files.

In [ ]:
from src.preprocessing.preprocess import preprocess

manifest_df = preprocess(
    csv_path=CONVERTED_CSV,
    out_dir=OUTPUT_DIR,
    raw_dir=RAW_PDB_DIR,
    esm_cache=ESM_CACHE_DIR,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    max_workers=MAX_WORKERS,
)

In [ ]:
# Inspect manifest
print("Manifest Summary")
print("=" * 40)
print(f"Total: {len(manifest_df)}")
print(f"\nStatus:")
print(manifest_df["status"].value_counts())

ok_mask = manifest_df["status"].str.startswith("ok")
print(f"\nSample entries:")
manifest_df[ok_mask][["pdb_id", "chain", "split", "cv_batch", "n_residues", "n_binding"]].head()

## 5. Create PyTorch Datasets

In [ ]:
from src.preprocessing.preprocess import LazyGraphDataset, get_cv_datasets

MANIFEST_PATH = Path(OUTPUT_DIR) / "manifest.csv"

train_dataset, val_dataset, test_dataset = get_cv_datasets(
    manifest_path=str(MANIFEST_PATH),
    cv_fold=CV_FOLD,
)

print(f"Dataset sizes (CV fold {CV_FOLD}):")
print(f"  Train: {len(train_dataset)}")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")

In [ ]:
# Inspect a graph
sample = train_dataset[0]

print("Sample Graph")
print("=" * 40)
print(f"x (node features): {sample.x.shape}")
print(f"edge_index:        {sample.edge_index.shape}")
print(f"edge_attr:         {sample.edge_attr.shape}")
print(f"pos:               {sample.pos.shape}")
print(f"y (labels):        {sample.y.shape}")
print(f"Binding residues:  {int(sample.y.sum())}")

## 6. Create DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=(DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(DEVICE == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(DEVICE == "cuda"),
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Verify batching
batch = next(iter(train_loader))

print("Sample Batch")
print("=" * 40)
print(f"x shape:          {batch.x.shape}")
print(f"edge_index shape: {batch.edge_index.shape}")
print(f"batch tensor:     {batch.batch.shape}")
print(f"Graphs in batch:  {batch.num_graphs}")

## 7. Dataset Statistics

In [ ]:
def compute_stats(dataset, name, n_samples=100):
    n_nodes, n_edges, n_binding = [], [], []
    
    for i in range(min(len(dataset), n_samples)):
        g = dataset[i]
        n_nodes.append(g.num_nodes)
        n_edges.append(g.edge_index.shape[1])
        n_binding.append(int(g.y.sum()))
    
    binding_ratio = np.mean([b/n for b, n in zip(n_binding, n_nodes)])
    
    print(f"\n{name} (sampled {len(n_nodes)})")
    print(f"  Nodes:   {min(n_nodes)}-{max(n_nodes)}, mean={np.mean(n_nodes):.0f}")
    print(f"  Edges:   {min(n_edges)}-{max(n_edges)}, mean={np.mean(n_edges):.0f}")
    print(f"  Binding: {min(n_binding)}-{max(n_binding)}, mean={np.mean(n_binding):.1f}")
    print(f"  Binding ratio: {binding_ratio*100:.2f}%")

compute_stats(train_dataset, "Training")
compute_stats(val_dataset, f"Validation (fold {CV_FOLD})")
compute_stats(test_dataset, "Test")

## 8. Save Metadata

In [ ]:
metadata = {
    "csv_path": str(CONVERTED_CSV),
    "output_dir": str(OUTPUT_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "device": DEVICE,
    "cv_fold": CV_FOLD,
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "test_size": len(test_dataset),
    "node_feature_dim": sample.x.shape[1],
    "edge_attr_dim": sample.edge_attr.shape[1],
}

metadata_path = Path(OUTPUT_DIR) / "metadata.pt"
torch.save(metadata, metadata_path)
print(f"Saved metadata to {metadata_path}")

for k, v in metadata.items():
    print(f"  {k}: {v}")

## 9. Training Sanity Check

In [ ]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class SimpleGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)
    
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.lin(x).squeeze(-1)

model = SimpleGNN(sample.x.shape[1], 128).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()

model.train()
total_loss = 0

for i, batch in enumerate(train_loader):
    if i >= 5:
        break
    
    batch = batch.to(DEVICE)
    optimizer.zero_grad()
    out = model(batch.x, batch.edge_index, batch.batch)
    loss = criterion(out, batch.y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

print(f"Sanity check passed!")
print(f"Avg loss (5 batches): {total_loss/5:.4f}")

## Summary

Output structure:
```
preprocessed/
├── train/cv_1/ ... cv_5/  (.pt files)
├── test/                   (.pt files)
├── pdb_test/               (PDB files)
├── manifest.csv
└── metadata.pt
```

Usage:
```python
from src.preprocessing.preprocess import get_cv_datasets
train_ds, val_ds, test_ds = get_cv_datasets("preprocessed/manifest.csv", cv_fold=1)
```